Using the same features (but just their means) as the paper we can get a C-index of ~ 0.607. The model is correctly predicting injuries better than a random guess.


In [13]:
from lifelines import CoxPHFitter, KaplanMeierFitter
import pandas as pd

df = df.sort_values(['Athlete ID', 'Date']).reset_index(drop=True)

baseline_data = []

for athlete in df['Athlete ID'].unique():
    athlete_df = df[df['Athlete ID'] == athlete]
    
    # Time: number of days observed
    T = len(athlete_df)
    
    # Event: did injury occur
    E = int(athlete_df['injury'].sum() > 0)
    
    total_km_mean = athlete_df['total km'].mean()
    perceived_exertion_mean = athlete_df['perceived exertion'].mean()
    
    baseline_data.append({
        'athlete_id': athlete,
        'days_observed': T,
        'injured': E,
        'avg_total_km': total_km_mean,
        'avg_exertion': perceived_exertion_mean,
    })

baseline_df = pd.DataFrame(baseline_data)

cph = CoxPHFitter()
cph.fit(baseline_df[['avg_total_km', 'avg_exertion', 'days_observed', 'injured']], 
        duration_col='days_observed', event_col='injured')

print(cph.summary)
print(f"C-index: {cph.concordance_index_:.3f}")

                  coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  \
covariate                                                                     
avg_total_km -0.060688   0.941116  0.041920       -0.142850        0.021473   
avg_exertion -2.888673   0.055650  1.243883       -5.326639       -0.450707   

              exp(coef) lower 95%  exp(coef) upper 95%  cmp to         z  \
covariate                                                                  
avg_total_km             0.866884             1.021706     0.0 -1.447718   
avg_exertion             0.004860             0.637177     0.0 -2.322303   

                     p  -log2(p)  
covariate                         
avg_total_km  0.147696  2.759297  
avg_exertion  0.020217  5.628314  
C-index: 0.607
